# 09 Train Edge MLP to Predict Posterior Expected Utility

This notebook trains a scalable edge-level MLP to predict posterior expected utility, `EU`, from the same pre-exposure edge features used in notebook 03.

The target comes from notebook 07:

$$
EU_{itj}=E\left[u_{itj}\mid \text{watch ratio}_{itj},x_{itj},B_{it},c_i,\hat\theta,\hat\phi\right].
$$

Important design choices:

- Train only an edge MLP. No GNN/message passing in this notebook.
- Reuse notebook-03 edge features: `x_cont`, `x_bin`, and `x_cat` from notebook 02's `gnn_data.pt` artifact.
- Do not use structural quantities as input features: no `mu_hat`, `sigma_hat`, `tau_hat`, `r_hat`, `q_target`, `u_star`, or `EU` as features.
- Split sessions chronologically into 80/10/10 train/validation/test among rows with valid `EU` labels.
- Train on standardized `EU` using train-split mean and standard deviation, then report predictions back on the original utility scale.

## Feature Dictionary

The edge MLP uses only the pre-exposure edge features saved by notebook 02 and reused by notebook 03. These features are stored in three aligned tensors: `x_cont`, `x_bin`, and `x_cat`. Each row corresponds to the same interaction edge as `edge_index[:, row_id]`.

The model does **not** use posterior or structural-estimation quantities as inputs. Excluded variables include `EU`, `q_target`, `r_hat`, `tau_hat`, `mu_hat`, `sigma_hat`, `u_star`, and `tilde_r`.

| Tensor | Feature | Description / construction |
|---|---|---|
| `x_cont` | `u_follow_user_num` | User profile count: number of accounts the user follows. |
| `x_cont` | `u_fans_user_num` | User profile count: number of fans/followers. |
| `x_cont` | `u_friend_user_num` | User profile count: number of friends. |
| `x_cont` | `u_register_days` | User account age in days. |
| `x_cont` | `u_follow_user_num_log1p` | \(\log(1+\text{follow count})\), included to reduce skew. |
| `x_cont` | `u_fans_user_num_log1p` | \(\log(1+\text{fan count})\), included to reduce skew. |
| `x_cont` | `u_friend_user_num_log1p` | \(\log(1+\text{friend count})\), included to reduce skew. |
| `x_cont` | `u_register_days_log1p` | \(\log(1+\text{registration days})\), included to reduce skew. |
| `x_cont` | `i_aspect_ratio` | Item metadata: video aspect ratio. |
| `x_cont` | `i_video_duration` | Item metadata: video duration. |
| `x_cont` | `i_age_since_upload_days` | Item-context feature: days between upload time and the current interaction time. |
| `x_cont` | `ctx_hour_sin` | Time-context feature: sine encoding of hour of day. |
| `x_cont` | `ctx_hour_cos` | Time-context feature: cosine encoding of hour of day. |
| `x_cont` | `hist_ema_y_complete` | Shifted user history: exponential moving average of prior completion outcomes before the current interaction. |
| `x_cont` | `hist_ema_y_long` | Shifted user history: exponential moving average of prior long-watch outcomes before the current interaction. |
| `x_cont` | `hist_ema_y_rewatch` | Shifted user history: exponential moving average of prior rewatch outcomes before the current interaction. |
| `x_cont` | `hist_ema_y_neg` | Shifted user history: exponential moving average of prior very-short-watch outcomes before the current interaction. |
| `x_cont` | `hist_ema_watchratio` | Shifted user history: exponential moving average of prior watch ratio before the current interaction. |
| `x_cont` | `hist_cat_ema_complete` | Shifted user-category history: prior completion EMA for this user and current level-1 category. |
| `x_cont` | `hist_cat_entropy_l2` | Shifted user history: entropy of the user's historical level-2 category mix, normalized by \(\log\) number of categories. |
| `x_cont` | `hist_author_recency_days` | Shifted user-author history: days since this user last saw the same author. |
| `x_cont` | `hist_prev_sess_len` | Shifted user history: length of the previous completed session. |
| `x_cont` | `hist_intersess_gap_h` | Shifted user history: hours since the previous session ended. |
| `x_bin` | `u_is_lowactive_period` | User profile flag for low-activity period. |
| `x_bin` | `u_is_live_streamer` | User profile flag for whether the user is a live streamer. |
| `x_bin` | `u_is_video_author` | User profile flag for whether the user is a video author. |
| `x_bin` | `ctx_is_weekend` | Time-context flag for weekend interaction. |
| `x_bin` | `hist_last_complete_author` | Shifted user-author history: whether the last exposure to this author was completed. |
| `x_bin` | `hist_has_author_history` | Shifted user-author history: whether this user has previously seen the same author. |
| `x_cat` | `burst_id` | Dense activity-burst identifier encoded as a category. |
| `x_cat` | `session` | User session identifier encoded as a category. |
| `x_cat` | `u_user_active_degree` | User profile active-degree bucket. |
| `x_cat` | `u_follow_user_num_range` | User profile bucket for followed-user count. |
| `x_cat` | `u_fans_user_num_range` | User profile bucket for fan/follower count. |
| `x_cat` | `u_friend_user_num_range` | User profile bucket for friend count. |
| `x_cat` | `u_register_days_range` | User profile bucket for account age. |
| `x_cat` | `u_onehot_feat0-17` | Encrypted user categorical feature 0-17, stored as a categorical code. |
| `x_cat` | `i_author_id` | Item metadata: video author ID. |
| `x_cat` | `i_video_type` | Item metadata: video type. |
| `x_cat` | `i_upload_type` | Item metadata: upload type. |
| `x_cat` | `i_visible_status` | Item metadata: visibility status. |
| `x_cat` | `i_music_id` | Item metadata: music ID. |
| `x_cat` | `i_video_tag_id` | Item metadata: video tag ID. |
| `x_cat` | `i_video_tag_name` | Item metadata: video tag name. |
| `x_cat` | `i_cat_level1_id` | Item metadata: level-1 content category ID. |
| `x_cat` | `i_cat_level2_id` | Item metadata: level-2 content category ID. |
| `x_cat` | `i_cat_level3_id` | Item metadata: level-3 content category ID. |

## 1. Setup

In [1]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"
DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "gnn_data.pt"
PROCESSED_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "processed_data.parquet"
EU_TARGET_PATH = OUTPUT_DIR / "semi_synth_observed_targets.parquet"
UTILITY_OBSERVED_PATH = OUTPUT_DIR / "semi_synth_utility_observed.parquet"

PREDICTION_OUTPUT_PATH = OUTPUT_DIR / "eu_edge_mlp_predictions.parquet"
METRICS_OUTPUT_PATH = OUTPUT_DIR / "eu_edge_mlp_metrics.json"
CHECKPOINT_OUTPUT_PATH = OUTPUT_DIR / "eu_edge_mlp_model.pt"
LOSS_HISTORY_OUTPUT_PATH = OUTPUT_DIR / "eu_edge_mlp_loss_history.csv"

SEED = 20260623
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

TRAIN_SESSION_SHARE = 0.80
VALID_SESSION_SHARE = 0.10
TEST_SESSION_SHARE = 0.10

BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 131072
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 256
DROPOUT = 0.10
LOSS_NAME = "mse"  # choices: "mse", "smooth_l1"
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-5
NUM_WORKERS = 0

required_paths = [DATA_PATH, PROCESSED_DATA_PATH, EU_TARGET_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)

device: mps


## 2. Load Notebook-02/03 Feature Artifact

In [2]:
data = torch.load(DATA_PATH, map_location="cpu", weights_only=False)
required_keys = ["edge_index", "x_cont", "x_bin", "x_cat", "y", "feature_metadata"]
missing = [k for k in required_keys if k not in data]
if missing:
    raise KeyError(f"Missing keys in gnn_data.pt: {missing}")

metadata = data["feature_metadata"]
continuous_cols = metadata["continuous_cols"]
binary_cols = metadata["binary_cols"]
categorical_cols = metadata["categorical_cols"]
categorical_cardinalities = metadata.get("categorical_cardinalities")
if categorical_cardinalities is None:
    categorical_cardinalities = (data["x_cat"].max(dim=0).values + 1).tolist()
categorical_cardinalities = [int(c) for c in categorical_cardinalities]

n_edges = int(data["edge_index"].shape[1])
checks = {
    "x_cont_rows": data["x_cont"].shape[0] == n_edges,
    "x_bin_rows": data["x_bin"].shape[0] == n_edges,
    "x_cat_rows": data["x_cat"].shape[0] == n_edges,
    "x_cont_width": data["x_cont"].shape[1] == len(continuous_cols),
    "x_bin_width": data["x_bin"].shape[1] == len(binary_cols),
    "x_cat_width": data["x_cat"].shape[1] == len(categorical_cols),
    "categorical_cardinality_width": len(categorical_cardinalities) == len(categorical_cols),
}
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise ValueError(f"Feature artifact validation failed: {failed}")

for forbidden in ["EU", "q_target", "r_hat", "tau_hat", "mu_hat", "sigma_hat", "u_star", "tilde_r"]:
    if forbidden in continuous_cols or forbidden in binary_cols or forbidden in categorical_cols:
        raise ValueError(f"Forbidden structural/target feature found in edge features: {forbidden}")

print("edges:", n_edges)
print("feature dims:", {"continuous": len(continuous_cols), "binary": len(binary_cols), "categorical": len(categorical_cols)})
print("first continuous features:", continuous_cols[:8])
print("first categorical features:", categorical_cols[:8])
display(pd.DataFrame({
    "group": ["continuous", "binary", "categorical"],
    "n_features": [len(continuous_cols), len(binary_cols), len(categorical_cols)],
}))

edges: 12527912
feature dims: {'continuous': 23, 'binary': 6, 'categorical': 35}
first continuous features: ['u_follow_user_num', 'u_fans_user_num', 'u_friend_user_num', 'u_register_days', 'u_follow_user_num_log1p', 'u_fans_user_num_log1p', 'u_friend_user_num_log1p', 'u_register_days_log1p']
first categorical features: ['burst_id', 'session', 'u_user_active_degree', 'u_follow_user_num_range', 'u_fans_user_num_range', 'u_friend_user_num_range', 'u_register_days_range', 'u_onehot_feat0']


,group,n_features
0,continuous,23
1,binary,6
2,categorical,35


## 3. Load EU Labels and Build Chronological Session Split

In [3]:
target_cols = [
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx",
    "EU", "q_target", "log_watch_ratio",
]
target_df = pd.read_parquet(EU_TARGET_PATH, columns=target_cols)
target_df = target_df.loc[np.isfinite(target_df["EU"].to_numpy(dtype=np.float64))].copy()
target_df["row_id"] = target_df["row_id"].astype(np.int64)
if target_df["row_id"].min() < 0 or target_df["row_id"].max() >= n_edges:
    raise ValueError("row_id is outside gnn_data edge range")
if target_df["row_id"].duplicated().any():
    raise ValueError("Duplicate row_id in EU target table")

proc_cols = ["user_id", "video_id", "burst_id", "session", "time", "watch_ratio", "y_complete"]
processed = pd.read_parquet(PROCESSED_DATA_PATH, columns=proc_cols)
row_id_np = target_df["row_id"].to_numpy(dtype=np.int64)
proc_rows = processed.iloc[row_id_np].reset_index(drop=True)
proc_rows["row_id"] = row_id_np

for key in ["user_id", "video_id", "burst_id", "session"]:
    if not np.array_equal(target_df[key].to_numpy(), proc_rows[key].to_numpy()):
        raise ValueError(f"Target rows are not aligned with processed_data for key {key}")

labeled_df = target_df.reset_index(drop=True).merge(
    proc_rows[["row_id", "time", "watch_ratio", "y_complete"]],
    on="row_id",
    how="left",
    validate="one_to_one",
)
labeled_df["time"] = pd.to_datetime(labeled_df["time"])
labeled_df["session_key"] = (
    labeled_df["user_id"].astype(str) + "||" +
    labeled_df["burst_id"].astype(str) + "||" +
    labeled_df["session"].astype(str)
)

session_df = (
    labeled_df
    .groupby(["session_key", "user_id", "burst_id", "session"], observed=True)
    .agg(
        session_start_time=("time", "min"),
        session_end_time=("time", "max"),
        n_interactions=("row_id", "size"),
        first_row_id=("row_id", "min"),
    )
    .reset_index()
    .sort_values(["session_start_time", "first_row_id"], kind="mergesort")
    .reset_index(drop=True)
)

n_sessions = len(session_df)
train_session_end = int(round(n_sessions * TRAIN_SESSION_SHARE))
valid_session_end = int(round(n_sessions * (TRAIN_SESSION_SHARE + VALID_SESSION_SHARE)))
train_keys = set(session_df.iloc[:train_session_end]["session_key"])
valid_keys = set(session_df.iloc[train_session_end:valid_session_end]["session_key"])
test_keys = set(session_df.iloc[valid_session_end:]["session_key"])

session_split_map = {key: "train" for key in train_keys}
session_split_map.update({key: "valid" for key in valid_keys})
session_split_map.update({key: "test" for key in test_keys})
labeled_df["split"] = labeled_df["session_key"].map(session_split_map)
if labeled_df["split"].isna().any():
    raise RuntimeError("Some labeled rows were not assigned to a split")

split_counts = labeled_df["split"].value_counts().reindex(["train", "valid", "test"])
session_split_counts = pd.Series({"train": len(train_keys), "valid": len(valid_keys), "test": len(test_keys)})
print("labeled rows:", len(labeled_df))
print("sessions:", n_sessions)
print("session split counts:")
display(session_split_counts.to_frame("n_sessions"))
print("row split counts:")
display(split_counts.to_frame("n_rows"))
print("split date ranges:")
display(labeled_df.groupby("split", observed=True)["time"].agg(["min", "max", "count"]).loc[["train", "valid", "test"]])

labeled rows: 4325560
sessions: 81566
session split counts:


,n_sessions
train,65253
valid,8156
test,8157


row split counts:


,n_rows
split,
train,3987463
valid,213175
test,124922


split date ranges:


,min,max,count
split,,,
train,2020-07-05 14:26:32.553,2020-09-02 12:46:08.686,3987463
valid,2020-09-01 19:56:05.398,2020-09-04 15:26:34.140,213175
test,2020-09-04 00:11:39.227,2020-09-05 23:59:47.475,124922


## 4. Prepare Tensors and Train-Only Normalization

In [4]:
edge_ids_all_np = labeled_df["row_id"].to_numpy(dtype=np.int64)
edge_ids_by_split_np = {
    split: labeled_df.loc[labeled_df["split"].eq(split), "row_id"].to_numpy(dtype=np.int64)
    for split in ["train", "valid", "test"]
}

EU_np = np.full(n_edges, np.nan, dtype=np.float32)
EU_np[edge_ids_all_np] = labeled_df["EU"].to_numpy(dtype=np.float32)
train_edge_ids_np = edge_ids_by_split_np["train"]
EU_train = EU_np[train_edge_ids_np]
EU_train_mean = float(np.nanmean(EU_train))
EU_train_std = float(np.nanstd(EU_train))
if not np.isfinite(EU_train_std) or EU_train_std <= 0:
    raise ValueError("EU_train_std must be positive")

y_std_np = np.full(n_edges, np.nan, dtype=np.float32)
y_std_np[edge_ids_all_np] = ((EU_np[edge_ids_all_np] - EU_train_mean) / EU_train_std).astype(np.float32)

# Notebook 02 already standardized x_cont under its original split. Re-standardize
# the saved continuous tensor again using only this notebook's EU train split.
x_cont = data["x_cont"].float()
cont_mean = x_cont[torch.as_tensor(train_edge_ids_np, dtype=torch.long)].mean(dim=0)
cont_std = x_cont[torch.as_tensor(train_edge_ids_np, dtype=torch.long)].std(dim=0).clamp_min(1e-6)

x_bin = data["x_bin"].float()
x_cat = data["x_cat"].long()
y_std = torch.from_numpy(y_std_np).float()

split_edge_ids = {
    split: torch.from_numpy(ids).long()
    for split, ids in edge_ids_by_split_np.items()
}

print("EU train mean/std:", EU_train_mean, EU_train_std)
print("x_cont train mean abs after re-standardization should be checked in batches")
print("edge ids by split:", {k: int(v.numel()) for k, v in split_edge_ids.items()})

EU train mean/std: 2.1654117107391357 1.9293439388275146
x_cont train mean abs after re-standardization should be checked in batches
edge ids by split: {'train': 3987463, 'valid': 213175, 'test': 124922}


## 5. Define Edge MLP Regression Model

In [5]:
def default_embedding_dim(cardinality):
    return min(32, max(2, int(round(float(cardinality) ** 0.25 * 4))))


class EdgeFeatureEncoder(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, embedding_dims=None):
        super().__init__()
        self.continuous_dim = int(continuous_dim)
        self.binary_dim = int(binary_dim)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]
        if embedding_dims is None:
            embedding_dims = [default_embedding_dim(c) for c in self.categorical_cardinalities]
        self.embedding_dims = [int(d) for d in embedding_dims]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])
        self.output_dim = self.continuous_dim + self.binary_dim + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.continuous_dim:
            parts.append(x_cont_batch.float())
        if self.binary_dim:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=-1)


class EdgeMLPRegressor(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.edge_encoder = EdgeFeatureEncoder(categorical_cardinalities, continuous_dim, binary_dim)
        self.net = nn.Sequential(
            nn.Linear(self.edge_encoder.output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        edge_repr = self.edge_encoder(x_cont_batch, x_bin_batch, x_cat_batch)
        return self.net(edge_repr).squeeze(-1)


model = EdgeMLPRegressor(
    categorical_cardinalities=categorical_cardinalities,
    continuous_dim=len(continuous_cols),
    binary_dim=len(binary_cols),
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.MSELoss() if LOSS_NAME == "mse" else nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)
print("encoder output dim:", model.edge_encoder.output_dim)
print("parameters:", sum(p.numel() for p in model.parameters()))

EdgeMLPRegressor(
  (edge_encoder): EdgeFeatureEncoder(
    (embeddings): ModuleList(
      (0): Embedding(4, 6)
      (1): Embedding(241, 16)
      (2): Embedding(5, 6)
      (3): Embedding(9, 7)
      (4-6): 3 x Embedding(8, 7)
      (7): Embedding(3, 5)
      (8): Embedding(9, 7)
      (9): Embedding(31, 9)
      (10): Embedding(1077, 23)
      (11): Embedding(14, 8)
      (12): Embedding(11, 7)
      (13): Embedding(4, 6)
      (14): Embedding(48, 11)
      (15): Embedding(341, 17)
      (16): Embedding(8, 7)
      (17): Embedding(6, 6)
      (18-24): 7 x Embedding(4, 6)
      (25): Embedding(7408, 32)
      (26): Embedding(3, 5)
      (27): Embedding(19, 8)
      (28): Embedding(3, 5)
      (29): Embedding(7664, 32)
      (30): Embedding(547, 19)
      (31): Embedding(511, 19)
      (32): Embedding(40, 10)
      (33): Embedding(138, 14)
      (34): Embedding(215, 15)
    )
  )
  (net): Sequential(
    (0): Linear(in_features=392, out_features=256, bias=True)
    (1): ReLU()
    (2

## 6. Training and Evaluation Helpers

In [6]:
def make_loader(edge_ids_tensor, batch_size, shuffle):
    dataset = TensorDataset(edge_ids_tensor)
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        generator=generator if shuffle else None,
    )


def batch_features(edge_ids):
    edge_ids_cpu = edge_ids.cpu()
    xb_cont = ((x_cont[edge_ids_cpu] - cont_mean) / cont_std).to(DEVICE)
    xb_bin = x_bin[edge_ids_cpu].to(DEVICE)
    xb_cat = x_cat[edge_ids_cpu].to(DEVICE)
    yb = y_std[edge_ids_cpu].to(DEVICE)
    return xb_cont, xb_bin, xb_cat, yb


def inverse_standardize(y_std_values):
    return y_std_values * EU_train_std + EU_train_mean


def corr_safe(x, y, method):
    s = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["x"].nunique() < 2 or s["y"].nunique() < 2:
        return float("nan")
    return float(s["x"].corr(s["y"], method=method))


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    err = y_pred - y_true
    return {
        "n": int(len(y_true)),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "pearson": corr_safe(y_true, y_pred, "pearson"),
        "spearman": corr_safe(y_true, y_pred, "spearman"),
    }


@torch.no_grad()
def predict_split(split):
    model.eval()
    loader = make_loader(split_edge_ids[split], EVAL_BATCH_SIZE, shuffle=False)
    pred_std_parts = []
    target_std_parts = []
    edge_parts = []
    losses = []
    n_total = 0
    for (edge_ids,) in loader:
        xb_cont, xb_bin, xb_cat, yb = batch_features(edge_ids)
        pred = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(pred, yb)
        n = int(edge_ids.numel())
        losses.append(float(loss.item()) * n)
        n_total += n
        pred_std_parts.append(pred.detach().cpu().numpy())
        target_std_parts.append(yb.detach().cpu().numpy())
        edge_parts.append(edge_ids.cpu().numpy())
    pred_std = np.concatenate(pred_std_parts)
    target_std = np.concatenate(target_std_parts)
    edge_ids_np = np.concatenate(edge_parts)
    pred_eu = inverse_standardize(pred_std)
    target_eu = inverse_standardize(target_std)
    metrics = regression_metrics(target_eu, pred_eu)
    metrics["loss_std"] = sum(losses) / max(n_total, 1)
    return edge_ids_np, target_eu, pred_eu, metrics

## 7. Train Edge MLP

In [7]:
train_loader = make_loader(split_edge_ids["train"], BATCH_SIZE, shuffle=True)
loss_history = []
best_val_rmse = float("inf")
best_state = None
best_epoch = 0
stale_epochs = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_n = 0
    for (edge_ids,) in train_loader:
        xb_cont, xb_bin, xb_cat, yb = batch_features(edge_ids)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        n = int(edge_ids.numel())
        train_loss_sum += float(loss.item()) * n
        train_n += n

    train_loss = train_loss_sum / max(train_n, 1)
    _, _, _, valid_metrics = predict_split("valid")
    record = {
        "epoch": int(epoch),
        "train_loss_std": float(train_loss),
        "valid_loss_std": float(valid_metrics["loss_std"]),
        "valid_mae": float(valid_metrics["mae"]),
        "valid_rmse": float(valid_metrics["rmse"]),
        "valid_pearson": float(valid_metrics["pearson"]),
        "valid_spearman": float(valid_metrics["spearman"]),
    }
    loss_history.append(record)
    print(record)

    if valid_metrics["rmse"] < best_val_rmse - MIN_DELTA:
        best_val_rmse = valid_metrics["rmse"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"early stopping at epoch {epoch}; best epoch {best_epoch}, best validation RMSE {best_val_rmse:.6g}")
            break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

loss_history_df = pd.DataFrame(loss_history)
loss_history_df.to_csv(LOSS_HISTORY_OUTPUT_PATH, index=False)
print("best epoch:", best_epoch)
print("saved loss history:", LOSS_HISTORY_OUTPUT_PATH)

{'epoch': 1, 'train_loss_std': 0.17891089609565325, 'valid_loss_std': 0.08179083826017083, 'valid_mae': 0.4087902283950918, 'valid_rmse': 0.5517749587855814, 'valid_pearson': 0.9574446879422172, 'valid_spearman': 0.9506978959457036}
{'epoch': 2, 'train_loss_std': 0.08203090058035006, 'valid_loss_std': 0.07159534179410804, 'valid_mae': 0.38431349415173494, 'valid_rmse': 0.5162404700677125, 'valid_pearson': 0.9629166863955485, 'valid_spearman': 0.9572572801160377}
{'epoch': 3, 'train_loss_std': 0.07451639938959978, 'valid_loss_std': 0.06827192739590651, 'valid_mae': 0.37504197186429433, 'valid_rmse': 0.5041162995758371, 'valid_pearson': 0.9647231771419323, 'valid_spearman': 0.9593023556869164}
{'epoch': 4, 'train_loss_std': 0.07086037240668527, 'valid_loss_std': 0.06615686262997399, 'valid_mae': 0.3743722501242302, 'valid_rmse': 0.496246114468576, 'valid_pearson': 0.9659338241939648, 'valid_spearman': 0.9607345493803531}
{'epoch': 5, 'train_loss_std': 0.06854181465743206, 'valid_loss_std

## 8. Evaluate and Save Predictions

In [8]:
prediction_frames = []
metrics = {
    "target": "EU",
    "model": "edge_mlp",
    "split_rule": "chronological session split among EU-labeled rows: 80/10/10",
    "config": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "loss_name": LOSS_NAME,
        "best_epoch": int(best_epoch),
        "EU_train_mean": EU_train_mean,
        "EU_train_std": EU_train_std,
        "device": DEVICE,
    },
    "splits": {},
}

for split in ["train", "valid", "test"]:
    edge_ids_np, target_eu, pred_eu, split_metrics = predict_split(split)
    metrics["splits"][split] = split_metrics
    frame = pd.DataFrame({
        "row_id": edge_ids_np.astype(np.int64),
        "split": split,
        "EU": target_eu.astype(np.float32),
        "pred_EU": pred_eu.astype(np.float32),
    })
    prediction_frames.append(frame)
    print(split, split_metrics)

pred_df = pd.concat(prediction_frames, ignore_index=True)
meta_cols = [
    "row_id", "user_id", "video_id", "burst_id", "session", "sess_idx", "time",
    "watch_ratio", "log_watch_ratio", "y_complete", "q_target",
]
meta_df = labeled_df[meta_cols].copy()
pred_df = pred_df.merge(meta_df, on="row_id", how="left", validate="one_to_one")

if UTILITY_OBSERVED_PATH.exists():
    utility_cols = ["row_id", "u_star", "raw_score", "q_omega"]
    utility_df = pd.read_parquet(UTILITY_OBSERVED_PATH, columns=utility_cols)
    pred_df = pred_df.merge(utility_df, on="row_id", how="left", validate="one_to_one")

pred_df.to_parquet(PREDICTION_OUTPUT_PATH, index=False)
print("saved predictions:", PREDICTION_OUTPUT_PATH)
display(pred_df.head())

train {'n': 3987463, 'mae': 0.3383735185069727, 'rmse': 0.46296345397264965, 'pearson': 0.9708146673048086, 'spearman': 0.9668247998125793, 'loss_std': 0.057580324300421606}
valid {'n': 213175, 'mae': 0.34817218334996103, 'rmse': 0.47794650217240664, 'pearson': 0.9682580538719054, 'spearman': 0.963573914750381, 'loss_std': 0.06136761933348447}
test {'n': 124922, 'mae': 0.3548467716650677, 'rmse': 0.48033536252172854, 'pearson': 0.9656797740439688, 'spearman': 0.9632110038316356, 'loss_std': 0.06198260560631752}
saved predictions: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/eu_edge_mlp_predictions.parquet


,row_id,split,EU,pred_EU,user_id,video_id,burst_id,session,sess_idx,time,watch_ratio,log_watch_ratio,y_complete,q_target,u_star,raw_score,q_omega
0,5150,train,3.092592,2.926930,3,6854,1,4,0,2020-07-07 06:02:05.629,1.373667,0.317484,1,0.297841,3.310092,0.098014,0.512853
1,5151,train,3.323300,3.245885,3,1963,1,4,0,2020-07-07 06:02:13.260,0.796246,-0.227847,0,0.213407,3.212863,-0.055274,0.111965
2,5152,train,3.203014,3.093935,3,6776,1,4,0,2020-07-07 06:04:51.726,1.210851,0.191324,1,0.262581,2.644236,-0.017219,-0.306571
3,5153,train,3.364444,3.238303,3,256,1,4,0,2020-07-07 06:13:38.231,0.693002,-0.366722,0,0.251200,2.748331,-0.068776,-0.314729
4,5154,train,3.364181,3.250604,3,5279,1,4,0,2020-07-07 06:20:03.222,0.496902,-0.699363,0,0.250958,3.212863,-0.055274,0.111965


## 9. Ranking/Utility Diagnostics

In [9]:
diagnostic_records = []
if "u_star" in pred_df.columns and pred_df["u_star"].notna().any():
    for split in ["train", "valid", "test"]:
        sub = pred_df.loc[pred_df["split"].eq(split)].copy()
        candidates = {
            "pred_EU": sub["pred_EU"],
            "EU": sub["EU"],
            "watch_ratio": sub["watch_ratio"],
            "log_watch_ratio": sub["log_watch_ratio"],
            "completion": sub["y_complete"],
        }
        if "raw_score" in sub.columns:
            candidates["raw_score"] = sub["raw_score"]
        if "q_omega" in sub.columns:
            candidates["q_omega"] = sub["q_omega"]
        for name, values in candidates.items():
            diagnostic_records.append({
                "split": split,
                "score": name,
                "pearson_with_u_star": corr_safe(values, sub["u_star"], "pearson"),
                "spearman_with_u_star": corr_safe(values, sub["u_star"], "spearman"),
                "n": int(sub[["u_star"]].join(values.rename("score")).dropna().shape[0]),
            })

utility_diag_df = pd.DataFrame(diagnostic_records)
if len(utility_diag_df):
    display(utility_diag_df.sort_values(["split", "spearman_with_u_star"], ascending=[True, False]))
    metrics["u_star_diagnostics"] = utility_diag_df.to_dict(orient="records")
else:
    print("No u_star diagnostics available; run notebook 08 first to create semi_synth_utility_observed.parquet.")
    metrics["u_star_diagnostics"] = []

METRICS_OUTPUT_PATH.write_text(json.dumps(metrics, indent=2))
torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "config": metrics["config"],
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "categorical_cardinalities": categorical_cardinalities,
    "cont_mean": cont_mean.cpu(),
    "cont_std": cont_std.cpu(),
    "EU_train_mean": EU_train_mean,
    "EU_train_std": EU_train_std,
}, CHECKPOINT_OUTPUT_PATH)

print("saved metrics:", METRICS_OUTPUT_PATH)
print("saved checkpoint:", CHECKPOINT_OUTPUT_PATH)
print(json.dumps(metrics["splits"], indent=2))

,split,score,pearson_with_u_star,spearman_with_u_star,n
14,test,pred_EU,0.905981,0.921742,124922
15,test,EU,0.889620,0.903528,124922
20,test,q_omega,0.444963,0.399685,124922
19,test,raw_score,0.270818,0.255658,124922
16,test,watch_ratio,0.011554,0.146589,124922
17,test,log_watch_ratio,0.210750,0.146588,124922
18,test,completion,0.025481,0.015472,124922
0,train,pred_EU,0.928465,0.930275,3987463
1,train,EU,0.911207,0.910387,3987463
6,train,q_omega,0.407266,0.394490,3987463


saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/eu_edge_mlp_metrics.json
saved checkpoint: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/eu_edge_mlp_model.pt
{
  "train": {
    "n": 3987463,
    "mae": 0.3383735185069727,
    "rmse": 0.46296345397264965,
    "pearson": 0.9708146673048086,
    "spearman": 0.9668247998125793,
    "loss_std": 0.057580324300421606
  },
  "valid": {
    "n": 213175,
    "mae": 0.34817218334996103,
    "rmse": 0.47794650217240664,
    "pearson": 0.9682580538719054,
    "spearman": 0.963573914750381,
    "loss_std": 0.06136761933348447
  },
  "test": {
    "n": 124922,
    "mae": 0.3548467716650677,
    "rmse": 0.48033536252172854,
    "pearson": 0.9656797740439688,
    "spearman": 0.9632110038316356,
    "loss_std": 0.06198260560631752
  }
}
